In [ ]:
from pathlib import Path
import pandas as pd
from google.colab import files

BASE_DIR = Path("/content/")
DATA_PROCESSED = BASE_DIR / "datos" / "procesados"

In [ ]:
# AVANCE 2 ARCGIS

BASE = Path('/content/datos')

carpetas = [
    'geo',
    'documentos/geo'
]

for c in carpetas:
  (BASE / c).mkdir(parents=True, exist_ok=True)

(BASE / 'README.md').touch()

print("----- Estructura Creada -----")
for p in sorted(BASE.rglob('*')):
  nivel = len(p.relative_to(BASE).parts) - 1
  print(' ' * nivel + p.name + ('/' if p.is_dir() else ''))

In [ ]:
# DEFINIR RUTA DEL CSV DEL DATASET LIMPIO

ruta = DATA_PROCESSED /'dataset_limpio.csv'

# DEFINIR RUTA DEL CSV DEL DATASET INTEGRADO

ruta_dem = DATA_PROCESSED /'dataset_integrado.csv'

# VERIFICAR QUE AMBOS EXISTEN

faltantes = [r for r in (ruta, ruta_dem) if not r.exists()]
if faltantes:
  raise FileNotFoundError(f"Faltan: {faltantes}")
else:
  print("Ambos archivos CSV existen y estan listos para usar!!!")

In [ ]:
# LEER CSV DEL DATASET LIMPIO

df = pd.read_csv(ruta)

# LEER CSV DEL DATASET INTEGRADO

df_dem = pd.read_csv(ruta_dem)

# IMPRIMIR INFORMACION DEL DATASET LIMPIO
print("\n---- Archivo DATASET LIMPIO ----")
print(f"Registros: {len(df)}")
print(f"\nColumnas: {df.columns.tolist()}")
print(df.head())


# IMPRIMIR INFORMACION DEL DATASET INTEGRADO
print("\n---- Archivo DATASET INTEGRADO----")
print(f"Registros: {len(df_dem)}")
print(f"\nColumnas: {df_dem.columns.tolist()}")
print(df_dem.head())

In [ ]:
# CONVERTIR COORDENADAS A NUMERO

df['latitud_restaurante'] = pd.to_numeric(df['latitud_restaurante'], errors='coerce')
df['longitud_restaurante'] = pd.to_numeric(df['longitud_restaurante'], errors='coerce')
df['latitud_ubicacion_entrega'] = pd.to_numeric(df['latitud_ubicacion_entrega'], errors='coerce')
df['longitud_ubicacion_entrega'] = pd.to_numeric(df['longitud_ubicacion_entrega'], errors='coerce')

# Dataset integrado
df_dem['Latitud_Restaurante'] = pd.to_numeric(df_dem['Latitud_Restaurante'], errors='coerce')
df_dem['Longitud_Restaurante'] = pd.to_numeric(df_dem['Longitud_Restaurante'], errors='coerce')
df_dem['Latitud_Ubicacion_Entrega'] = pd.to_numeric(df_dem['Latitud_Ubicacion_Entrega'], errors='coerce')
df_dem['Longitud_Ubicacion_Entrega'] = pd.to_numeric(df_dem['Longitud_Ubicacion_Entrega'], errors='coerce')

# IMPRIMIR EL DATASET CON LAS COORDENADAS CONVERTIDAS
print("\nCoordenadas convertidas a numeros -------")
print("---DATASET LIMPIO---")
print(df[['latitud_ubicacion_entrega','longitud_ubicacion_entrega','latitud_restaurante', 'longitud_restaurante']].head())
print("\n\n\n---DATASET INTEGRADO---")
print(df_dem[['Latitud_Ubicacion_Entrega','Longitud_Ubicacion_Entrega','Latitud_Restaurante', 'Longitud_Restaurante']].head())

In [ ]:
df['latitud_restaurante'] = pd.to_numeric(
    df['latitud_restaurante'], errors='coerce'
)

df['longitud_restaurante'] = pd.to_numeric(
    df['longitud_restaurante'], errors='coerce'
)

mask_valida = (
    df['latitud_restaurante'].between(6.5, 35.5) &
    df['longitud_restaurante'].between(68.0, 97.5)
)

print(f"Total de registros: {len(df)}")
print(f"Coordenadas válidas: {mask_valida.sum()}")
print(f"Coordenadas a revisar: {(~mask_valida).sum()}")

df = df.loc[mask_valida].copy().reset_index(drop=True)

print(f"Registros finales: {len(df)}")

In [ ]:
# Convertir a numérico
df['latitud_ubicacion_entrega'] = pd.to_numeric(
    df['latitud_ubicacion_entrega'], errors='coerce'
)

df['longitud_ubicacion_entrega'] = pd.to_numeric(
    df['longitud_ubicacion_entrega'], errors='coerce'
)

# Validar coordenadas dentro de India
mask_valida = (
    df['latitud_ubicacion_entrega'].between(6.5, 35.5) &
    df['longitud_ubicacion_entrega'].between(68.0, 97.5)
)

print(f"Total de registros: {len(df)}")
print(f"Coordenadas válidas: {mask_valida.sum()}")
print(f"Coordenadas a revisar: {(~mask_valida).sum()}")

# Dejar df solamente con las coordenadas válidas
df = df.loc[mask_valida].copy().reset_index(drop=True)

print(f"Registros finales: {len(df)}")

In [ ]:
# REVISAR NULOS EN DATASET LIMPIO
print("Nulos en dataset limpio: ")
print(df.isna().sum())

# REVISAR DUPLICADOS EN DATASET LIMPIO
print(f"Duplicados en dataset limpio: {df.duplicated().sum()}")

# REVISAR COORDENADAS REPETIDAS EN DATASET LIMPIO
duplicados_coords = df.duplicated(subset=['latitud_restaurante', 'longitud_restaurante']).sum()
print(f"Coordenadas repetidas: {duplicados_coords}")

#--------------------------------
# REVISAR NULOS EN DATASET INTEGRADO
print("\n\nNulos en dataset integrado: ")
print(df.isna().sum())

# REVISAR DUPLICADOS EN DATASET INTEGRADO
print(f"Duplicados en dataset integrado: {df.duplicated().sum()}")

# REVISAR COORDENADAS REPETIDAS EN DATASET INTEGRADO
duplicados_coords = df.duplicated(subset=['latitud_restaurante', 'longitud_restaurante']).sum()
print(f"Coordenadas repetidas: {duplicados_coords}")

In [ ]:
# EXPORTAR VERSION VALIDADA

# Corregimos la ruta eliminando el 'datos' duplicado ya que BASE = Path('/content/datos')
ruta_salida = BASE / 'geo' / 'repartos_india_validado_zomato.csv'

# Asegurar que el directorio de destino exista
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"Archivo validado guardado en: {ruta_salida}")

# DESCARGAR EL ARCHIVO
from google.colab import files
files.download(str(ruta_salida))